# Project Structure

# Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")
import re
from collections import Counter


# Load Dataset

In [2]:
df=pd.read_csv("C:\\Users\\Client\\Assessment_project\\Amazon_Reviews.csv", engine='python')

# Check the dataset.

In [3]:
df.head(1)

,Reviewer Name,Profile Link,Country,Review Count,Review Date,Rating,Review Title,Review Text,Date of Experience
0,Eugene ath,/users/66e8185ff1598352d6b3701a,US,1 review,2024-09-16T13:44:26.000Z,Rated 1 out of 5 stars,A Store That Doesn't Want to Sell Anything,"I registered on the website, tried to order a ...","September 16, 2024"


In [4]:
df.shape

(21214, 9)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21214 entries, 0 to 21213
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Reviewer Name       21214 non-null  object
 1   Profile Link        21163 non-null  object
 2   Country             21054 non-null  object
 3   Review Count        21055 non-null  object
 4   Review Date         21055 non-null  object
 5   Rating              21055 non-null  object
 6   Review Title        21055 non-null  object
 7   Review Text         21055 non-null  object
 8   Date of Experience  20947 non-null  object
dtypes: object(9)
memory usage: 1.5+ MB


In [6]:
df.columns

Index(['Reviewer Name', 'Profile Link', 'Country', 'Review Count',
       'Review Date', 'Rating', 'Review Title', 'Review Text',
       'Date of Experience'],
      dtype='object')

Expected columns are

# Data Cleaning

Missing values

In [7]:
df.isnull().sum()

Reviewer Name           0
Profile Link           51
Country               160
Review Count          159
Review Date           159
Rating                159
Review Title          159
Review Text           159
Date of Experience    267
dtype: int64

Fill missing review text.

In [8]:
df["Review Text"] = df["Review Text"].fillna("No Review")

Remove rows where Rating is missing.

In [9]:
df = df.dropna(subset=["Rating"])

In [10]:
df = df.dropna(subset=["Date of Experience"])

In [11]:
df = df.dropna(subset=["Country"])

In [12]:
df.isnull().sum()

Reviewer Name         0
Profile Link          0
Country               0
Review Count          0
Review Date           0
Rating                0
Review Title          0
Review Text           0
Date of Experience    0
dtype: int64

Convert Rating into Numbers

Ratings are stored like

In [13]:
df["Rating"] = df["Rating"].str.extract("(\d)").astype(int)

In [14]:
df["Rating"].value_counts()

Rating
1    13120
5     4447
4     1278
2     1227
3      874
Name: count, dtype: int64

# Clean Review Text

Creating cleaning function.

In [15]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"[^a-zA-Z\s]", "", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

Apply it.

In [16]:
df["Clean_Review"] = df["Review Text"].apply(clean_text)

df[["Review Text","Clean_Review"]].head()

,Review Text,Clean_Review
0,"I registered on the website, tried to order a ...",i registered on the website tried to order a l...
1,Had multiple orders one turned up and driver h...,had multiple orders one turned up and driver h...
2,I informed these reprobates that I WOULD NOT B...,i informed these reprobates that i would not b...
3,I have bought from Amazon before and no proble...,i have bought from amazon before and no proble...
4,If I could give a lower rate I would! I cancel...,if i could give a lower rate i would i cancell...


# Rule-Based Filtering



Critical Reviews = 1-star and 2-star reviews.

In [17]:
critical_reviews = df[df["Rating"] <= 2]

critical_reviews.shape

(14347, 10)

Display a few reviews.

In [18]:
critical_reviews[["Rating","Review Title","Review Text"]].head()

,Rating,Review Title,Review Text
0,1,A Store That Doesn't Want to Sell Anything,"I registered on the website, tried to order a ..."
1,1,Had multiple orders one turned up and…,Had multiple orders one turned up and driver h...
2,1,I informed these reprobates,I informed these reprobates that I WOULD NOT B...
3,1,Advertise one price then increase it on website,I have bought from Amazon before and no proble...
4,1,If I could give a lower rate I would,If I could give a lower rate I would! I cancel...


# Complaint Keyword Analysis

Creating stop words.

In [19]:
stop_words = {
    "the","and","this","that","with","have","from",
    "they","their","been","were","would","there",
    "about","your","very","when","what","into",
    "after","before","because","could","should",
    "amazon","review","stars","star","product",
    "service","order","customer"
}

Creating keyword function.

In [32]:
def top_keywords(texts, top_n=20):

    words = []

    for review in texts:

        for word in review.split():

            if len(word) > 3 and word not in stop_words:
                words.append(word)

    return Counter(words).most_common(top_n)

In [33]:
keywords = top_keywords(critical_reviews["Clean_Review"])

keywords

[('them', 6643),
 ('delivery', 6230),
 ('will', 6135),
 ('account', 4908),
 ('time', 4899),
 ('item', 4674),
 ('prime', 4488),
 ('refund', 4482),
 ('just', 4367),
 ('dont', 4347),
 ('then', 4046),
 ('never', 3898),
 ('money', 3736),
 ('even', 3647),
 ('back', 3607),
 ('items', 3569),
 ('delivered', 3442),
 ('again', 3384),
 ('told', 3282),
 ('days', 3267)]

# Select Three Critical Reviews

In [34]:
critical_reviews["Length"] = critical_reviews["Review Text"].str.len()

top3 = critical_reviews.sort_values(
    by="Length",
    ascending=False
).head(3)

top3[["Rating","Review Title","Review Text"]]

,Rating,Review Title,Review Text
17974,1,ANTI-SEMITIC CLUB COMPANY: Unethical and Not P...,This company has disabled merchants for absolu...
18045,1,AMAZON RUINED MY CHRISTMAS - DO NOT TRUST AMAZ...,In relation to Order # 205-8187368-2071530:On ...
15792,1,My Honest Opinion About Amazon.com!,My honest opinion.I have been an Amazon Prime ...


# Generative AI

Install

In [23]:
!pip install google-generativeai

Import

In [24]:
import google.generativeai as genai

API Key
Check if you actually have a Gemini API key

Go to:

https://aistudio.google.com/app/apikey

Sign in with your Google account.
Click Create API Key.
Copy the key.

In [25]:
genai.configure(api_key="Enter API Key")

Model

In [26]:
model = genai.GenerativeModel("models/gemini-3.6-flash")

Function

In [38]:
def generate_email(review):

    prompt = f"""
You are a professional customer support executive.

Write a short apology email.

Customer Review:
{review}

The email should

- apologize
- acknowledge the issue
- sound empathetic
- promise improvement
- be under 150 words.
- include Gagan Kumar J K in the place of your name
"""

    response = model.generate_content(prompt)

    return response.text

Generate emails.

In [28]:
for i, review in enumerate(top3["Review Text"],1):

    print("="*60)

    print("Review",i)

    print(review)

    print()

    print(generate_email(review))

    print("\n")

Review 1
This company has disabled merchants for absolutely no reason; My family and multiple businesses are Jewish run, this is our story.  That is right, Amazon and Amazon Pay, are both anti Semitic company's. So, if you are jewish, as I am and my mother;  had her beauty salon on the Amazon account. They harassed her for "Selling Drugs or Adult Services", which was a major insult, as she was working on her new design for selling retail products for her beauty salon. Towne Plaza Hair studio has been in business for 29 years! I have been in business about ten years now and we both have never had a charge back nor refund. They have their User Agreements and every right to "part company for no reason", but the both of us followed every rule and guideline, they looked for anything they could discriminate against when doing their reviews. Our new site looks so nice and clean, so we will get to make money than ever and it is best to not put your money in company's that are this dishonest an

In [29]:
import google.generativeai as genai

genai.configure(api_key="Use Your API key")

for model in genai.list_models():
    if "generateContent" in model.supported_generation_methods:
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

Top Complaint Keywords

late
broken
refund
damaged
support
delivery
quality

Top Complaint Keywords

late
broken
refund
damaged
support
delivery
quality

AI Generated Response 1

<email>

AI Generated Response 2

<email>

AI Generated Response 3

<email>

# README.md

# Customer Feedback Analysis and Automated Response

## Objective

Analyze customer reviews using Python and generate AI-powered apology emails.

## Dataset

Amazon Reviews Dataset

## Steps

1. Load dataset
2. Clean missing values
3. Clean review text
4. Filter ratings <=2
5. Find complaint keywords
6. Select top 3 detailed reviews
7. Generate apology emails using Google Gemini

## Libraries

- pandas
- numpy
- re
- collections
- google-generativeai

## Running

Install dependencies

pip install pandas numpy google-generativeai

Add your Gemini API Key inside the notebook.

Run all notebook cells.